Para comenzar con la práctica, empezamos viendo los distintos valores posibles para **Color** y **Spectral_Class**:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

# Fijamos la semilla
np.random.seed(100522245)  # Semilla con NIA

# Cargamos los datos
df = pd.read_csv('stars_data.csv')

# Ver valores únicos y cuántos hay de cada uno en la Clase Espectral
print("--- Recuento de Clase Espectral ---")
print(df['Spectral_Class'].value_counts())

print("\n--- Recuento de Colores ---")
# Ver valores únicos de Color
print(df['Color'].value_counts())

--- Recuento de Clase Espectral ---
Spectral_Class
M    111
B     46
O     40
A     19
F     17
K      6
G      1
Name: count, dtype: int64

--- Recuento de Colores ---
Color
Red                   112
Blue                   56
Blue-white             26
Blue White             10
yellow-white            8
White                   7
Blue white              4
Yellowish White         3
white                   3
Whitish                 2
Orange                  2
yellowish               2
Pale yellow orange      1
White-Yellow            1
Yellowish               1
Orange-Red              1
Blue-White              1
Name: count, dtype: int64


Observamos que en **Spectral_Class** los valores son los esperados y, a priori, no hay nada que modificar (salvo convertirlos en número). No obstante, en **Color** podemos ver que hay colores que son los mismos pero están escritos de manera diferente, por lo que el algoritmo los tomará como colores distintos, como por ejemplo *Blue-white*, *Blue White*, *Blue white* y *Blue-White*, que son el mismo color pero difieren en mayúsculas, minúsculas y guiones. Por ello, a continuación abordaremos este problema poniendo todos los colores en un formato común:


In [2]:
# Normalización inicial de Color: quitamos espacios raros, pasamos a minúsculas y quitamos guiones
df['Color'] = df['Color'].str.lower().str.replace('-', ' ').str.strip()

# Hacemos un diccionario de corrección (mapeo de nombres feos a nombres limpios)
# Incluimos todas las variantes obtenidas en value_counts y les asignamos una común
limpieza_colores = {
    'red': 'red',
    'blue': 'blue',
    'blue white': 'blue white',
    'blue-white': 'blue white',
    'yellow-white': 'yellow white',
    'white yellow': 'yellow white',
    'yellowish white': 'yellow white',
    'white': 'white',
    'whitish': 'white',
    'yellowish': 'yellow',
    'orange': 'orange',
    'orange red': 'orange red',
    'pale yellow orange': 'pale yellow orange'
}

# Aplicamos la limpieza usando .replace() para no borrar los que ya están bien
df['Color'] = df['Color'].replace(limpieza_colores)

# Comprobamos que no se repiten datos y todo está como esperábamos
print("--- Clase Espectral (Limpia) ---")
print(df['Spectral_Class'].value_counts())

print("\n--- Colores (Limpios y agrupados) ---")
print(df['Color'].value_counts())

# Verificamos si hay algún valor nulo después de la limpieza
if df[['Color', 'Spectral_Class']].isnull().values.any():
    print("\nHay valores nulos")
else:
    print("\nNo hay valores nulos")

--- Clase Espectral (Limpia) ---
Spectral_Class
M    111
B     46
O     40
A     19
F     17
K      6
G      1
Name: count, dtype: int64

--- Colores (Limpios y agrupados) ---
Color
red                   112
blue                   56
blue white             41
white                  12
yellow white           12
yellow                  3
orange                  2
pale yellow orange      1
orange red              1
Name: count, dtype: int64

No hay valores nulos


Como podemos observar, hemos conseguido poner todos los colores en un formato común, que consiste en que todos los colores estén en minúscula y sin guiones. Esto lo logramos usando **lower** para las minúsculas, **replace** para sustituir los guiones por espacios y **strip** por si hubiese algún espacio raro en los nombres. A continuación, creamos un diccionario para renombrar a los colores y usamos **replace** para sustituir los colores por su nuevo nombre. Finalmente, comprobamos que todo está correctamente hecho y comprobamos si han aparecido nulos (no han aparecido). Ahora debemos llevar a cabo la **codificación ordinal** (importante hacerla antes de **PCA**, pues este solo entiende de números):

In [3]:
# Definimos orden lógico para color de menor a mayor temperatura/energía
# Para ello, nos fijamos en la tabla del enunciado y las relaciones color-temperatura
orden_color = {
    'red': 0,
    'orange red': 1, 
    'orange': 2, 
    'pale yellow orange': 3, 
    'yellow': 4, 
    'white': 5,            
    'yellow white': 6,     
    'blue white': 7, 
    'blue': 8
}

# Definimos orden lógico para spectral_class de menor a mayor temperatura/energía
# Para ello, nos fijamos en la secuencia del enunciado y la mención de que O es la más caliente y M la más fría
orden_espectral = {
    'M': 0, 
    'K': 1, 
    'G': 2, 
    'F': 3, 
    'A': 4, 
    'B': 5, 
    'O': 6
}

# Aplicamos el mapeo para crear las columnas numéricas
df['Rango_Color'] = df['Color'].map(orden_color)
df['Rango_Espectral'] = df['Spectral_Class'].map(orden_espectral)


# Comprobamos que no se repiten datos y todo está como esperábamos
print("--- Clase Espectral (Numerada) ---")
print(df['Rango_Espectral'].value_counts())

print("\n--- Colores (Numerados) ---")
print(df['Rango_Color'].value_counts())

# Verificación final de nulos
print("Nulos en Rango_Color:", df['Rango_Color'].isnull().sum())
print("Nulos en Rango_Espectral:", df['Rango_Espectral'].isnull().sum())

--- Clase Espectral (Numerada) ---
Rango_Espectral
0    111
5     46
6     40
4     19
3     17
1      6
2      1
Name: count, dtype: int64

--- Colores (Numerados) ---
Rango_Color
0    112
8     56
7     41
5     12
6     12
4      3
2      2
3      1
1      1
Name: count, dtype: int64
Nulos en Rango_Color: 0
Nulos en Rango_Espectral: 0


Para la **codificación ordianl** comenzamos escribiendo dos diccionarios, uno para **Color** y otro para **Spectral_Class**. Ambas columnas son ordinales, pues sus atributos se pueden ordenar en función de la temperatura (es decir, si son más calientes o más fríos). Para ello, observamos la tabla dada en el enunciado y vemos que en **Color**, *red* es el más frío y *yellow white* es el más caliente. Debido a que faltan colores, nos informamos sobre los colores que suelen ser más calientes en tema de estrellas y vimos que el más caliente era *blue*. Para otros colores fuimos construyendo la jerarquía siguiendo el siguiente razonamiento: por ejemplo, *orange* es más frío que *yellow* por lo que *pale yellow orange* está entre ellos en la escala. Para la codificación de **Spectral_Class** nos fijamos nuevamente en el enunciado, que dice que este atributo sigue una secuencia determinada, siendo **O** la más caliente y **M** la más fría, por lo que ordenamos los distintos valores siguiendo el orden de la secuencia dada en el enunciado. Finalmente, comprobamos que todo está como esperábamos y verificamos que no existen nulos.